# 04 - App Seed Export

Exporta `Files/generated/app_seed.json` para reproducir la experiencia local de la Fabric App.

In [ ]:
import json

results = spark.table("gov_quality_result")
rules = spark.table("gov_quality_rule")
actions = spark.table("gov_action")
features = spark.table("customer_features")
scores = spark.table("ml_customer_score_current")

quality_rows = (
    results.join(rules.select("rule_id", "rule_name", "quality_dimension", "severity"), "rule_id")
    .orderBy("score")
    .limit(12)
)
summary_row = spark.table("gov_quality_run").orderBy("started_at", ascending=False).first()
summary = {
    "asset_id": summary_row["asset_id"],
    "global_score": summary_row["global_score"],
    "failed_rules": results.where("status = 'Failed'").count(),
    "open_actions": actions.where("status = 'Open'").count(),
}
seed = {
    "governance": {
        "summary": summary,
        "rules": [row.asDict(recursive=True) for row in quality_rows.collect()],
        "actions": [row.asDict(recursive=True) for row in actions.collect()],
    },
    "intelligence": {
        "metrics": {"model_name": "customer_engagement_baseline", "model_version": "fabric-notebook-001"},
        "top_customers": [row.asDict(recursive=True) for row in scores.orderBy("score", ascending=False).limit(20).collect()],
        "feature_sample": [row.asDict(recursive=True) for row in features.limit(20).collect()],
    },
}

mssparkutils.fs.mkdirs("Files/generated")
mssparkutils.fs.put("Files/generated/quality_summary.json", json.dumps(summary, indent=2, default=str), True)
mssparkutils.fs.put("Files/generated/app_seed.json", json.dumps(seed, indent=2, default=str, ensure_ascii=False), True)

print("Wrote Files/generated/quality_summary.json")
print("Wrote Files/generated/app_seed.json")